# 🖼️ MOSS-Image (v4)
### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

###

**MOSS-Image** is an open 5-billion-parameter text-to-image foundation model designed for photorealistic generation and bilingual visual text rendering. Paired with **Kromoss**, an aesthetic fine-tune, this application provides an interactive interface for creating artwork, character designs, and posters with readable typography.

### ✨ Key Capabilities:
- 🔤 **Bilingual Visual Typography:** Renders crisp, readable text and signage in both English and Chinese by enclosing exact phrases in quotation marks.
- 🎨 **Distinct Model Checkpoints:**
  - **MOSS-Image SFT:** Fine-tuned for strong prompt adherence, natural textures, and complex scene composition.
  - **Kromoss:** An aesthetic fine-tune with richer contrast, vibrant palettes, and stylized character rendering.
  - **MOSS-Image Base:** The un-distilled foundation model weights for broad creative exploration.
- ⚡ **8-Step Turbo Acceleration:** Integrated support for Kromoss's 8-step DMD turbo LoRA for fast, high-quality generation.
- 🧠 **Intelligent Prompt Upsampling:** Integrated prompt engineer mode that expands concise ideas into comprehensive scene captions.
- 🎛️ **Dual LoRA Blending + Multi-Source Downloader (v2):** Load up to two LoRAs simultaneously, with support for downloading extra LoRAs from **Hugging Face**, **Civitai**, **Google Drive**, or direct URLs.
- 📦 **Extra Model Downloader (v3):** Download additional MOSS-compatible diffusion checkpoints from the same sources; they appear in the Web UI **Checkpoint** dropdown.
- 🏷️ **Embedded Prompt & Metadata:** Automatically embeds A1111 / Civitai parameters (`parameters` chunk), upsampled caption, ComfyUI workflow graph, and EXIF comments into every saved PNG.
- ⚡ **Independent Turbo LoRA Toggle (v4):** The 8-step DMD Turbo LoRA (`DOWNLOAD_TURBO_LORA`) is separated from the Kromoss model checkpoint, allowing you to use 8-step Turbo acceleration with MOSS-Image SFT / Base without downloading the 5.16 GB Kromoss checkpoint.

In [ ]:
#@title 1️⃣ Install Dependencies & Setup Web App { display-mode: "form" }
import os, re, subprocess, sys, urllib.request

COMFY_DIR = "/content/ComfyUI"
APP_DIR = "/content/moss_ui"
os.makedirs(APP_DIR, exist_ok=True)

uv_bin_dir = os.path.expanduser("~/.local/bin")
if uv_bin_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{uv_bin_dir}:{os.environ.get('PATH', '')}"

def sh(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

if subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True).returncode != 0:
    raise RuntimeError("No GPU found. Use Runtime > Change runtime type > T4 GPU, then run again.")

print("📦 Installing system packages & standalone package manager...")
sh("apt-get -qq update && apt-get -qq install -y aria2 openssh-client curl > /dev/null")
if subprocess.run("command -v uv", shell=True).returncode != 0:
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")

if not os.path.isdir(f"{COMFY_DIR}/.git"):
    print("📦 Setting up core runtime repository...")
    sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}")
else:
    sh(f"git -C {COMFY_DIR} pull --ff-only")

print("⚡ Installing Python packages...")
reqs = open(f"{COMFY_DIR}/requirements.txt").read().splitlines()
keep = [l for l in reqs if not re.match(r"^\s*(torch|torchvision|torchaudio)\s*($|[<>=!~;#\[])", l, re.I)]
open("/content/comfy_requirements.txt", "w").write("\n".join(keep) + "\n")
sh("uv pip install --system -q -r /content/comfy_requirements.txt flask requests gdown pillow")

WORKFLOW_URL = "https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/main/Moss%20image%20.json"
dest_wf = f"{APP_DIR}/workflow_api.json"
print("⬇️ Fetching generation workflow...")
req = urllib.request.Request(WORKFLOW_URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp, open(dest_wf, "wb") as f:
    f.write(resp.read())

print("📝 Initializing application engine...")
APP_CODE = r'''import copy
import json
import os
import random
import uuid
import requests
from flask import Flask, Response, jsonify, request

COMFY = os.environ.get("COMFY_URL", "http://127.0.0.1:8188").rstrip("/")
HERE = os.path.dirname(os.path.abspath(__file__))
COMFY_DIR = os.environ.get("COMFY_DIR", "/content/ComfyUI")
CLIENT_ID = uuid.uuid4().hex

JOB_PARAMS = {}

def embed_png_metadata(img_path, meta_info, final_caption):
    if not os.path.isfile(img_path):
        return
    try:
        from PIL import Image, PngImagePlugin
        with Image.open(img_path) as im:
            w, h = im.size
            im_format = im.format or "PNG"
            im_copy = im.copy()
            existing_info = dict(getattr(im, "info", {}))
            existing_exif = im.getexif()

        p = meta_info.get("params", {})
        seed = meta_info.get("seed", "")
        wf = meta_info.get("workflow", {})
        raw_prompt = p.get("prompt", "").strip()
        negative = p.get("negative", "").strip()
        prompt_text = final_caption.strip() if final_caption and final_caption.strip() else raw_prompt

        lines = [prompt_text]
        if negative:
            lines.append(f"Negative prompt: {negative}")
        param_parts = [
            f"Steps: {p.get('steps', 25)}",
            f"Sampler: {p.get('sampler', 'er_sde')}",
            f"Schedule type: {p.get('scheduler', 'simple')}",
            f"CFG scale: {p.get('cfg', 2.5)}",
            f"Seed: {seed}",
            f"Size: {w}x{h}",
            f"Model: {p.get('checkpoint', 'moss_image')}",
        ]
        l1 = p.get("lora1")
        if l1 and l1 != "None":
            param_parts.append(f"LoRA 1: {l1} ({p.get('lora1_strength', 1.0)})")
        l2 = p.get("lora2")
        if l2 and l2 != "None":
            param_parts.append(f"LoRA 2: {l2} ({p.get('lora2_strength', 1.0)})")
        if p.get("upsampler"):
            param_parts.append("Prompt Upsampler: True")
            if raw_prompt and raw_prompt != prompt_text:
                param_parts.append(f"Original prompt: {raw_prompt}")
        if p.get("slg"):
            param_parts.append("Skip-Layer Guidance: True")

        lines.append(", ".join(param_parts))
        a1111_params = "\n".join(lines)

        pnginfo = PngImagePlugin.PngInfo()
        for k, v in existing_info.items():
            if isinstance(v, str):
                pnginfo.add_text(k, v)

        pnginfo.add_text("parameters", a1111_params)
        pnginfo.add_text("caption", prompt_text)
        pnginfo.add_text("Description", prompt_text)
        pnginfo.add_text("Comment", prompt_text)
        if wf:
            pnginfo.add_text("workflow", json.dumps(wf))

        try:
            exif = existing_exif if existing_exif is not None else im_copy.getexif()
            exif[0x010e] = prompt_text[:2000]
            exif[0x9286] = f"Prompt: {prompt_text}\nNegative: {negative}\nSeed: {seed}"
            im_copy.save(img_path, format=im_format, pnginfo=pnginfo, exif=exif)
        except Exception:
            im_copy.save(img_path, format=im_format, pnginfo=pnginfo)
    except Exception as e:
        print(f"Metadata embedding note: {e}")


with open(os.path.join(HERE, "workflow_api.json"), encoding="utf-8") as f:
    WORKFLOW = json.load(f)

DEFAULTS = {
    "prompt": WORKFLOW.get("18:17", {}).get("inputs", {}).get("value", ""),
    "negative": WORKFLOW.get("18:5", {}).get("inputs", {}).get("text", ""),
    "steps": WORKFLOW.get("18:7", {}).get("inputs", {}).get("steps", 25),
    "cfg": WORKFLOW.get("18:7", {}).get("inputs", {}).get("cfg", 2.5),
    "sampler": WORKFLOW.get("18:7", {}).get("inputs", {}).get("sampler_name", "er_sde"),
    "scheduler": WORKFLOW.get("18:7", {}).get("inputs", {}).get("scheduler", "simple"),
    "aspect_ratio": WORKFLOW.get("19", {}).get("inputs", {}).get("aspect_ratio", "1:1 (Square)"),
    "megapixels": WORKFLOW.get("19", {}).get("inputs", {}).get("megapixels", 1),
    "checkpoint": WORKFLOW.get("18:1", {}).get("inputs", {}).get("unet_name", "moss_image_sft_int8_convrot.safetensors"),
    "upsampler": bool(WORKFLOW.get("18:16", {}).get("inputs", {}).get("switch", True)),
    "slg": bool(WORKFLOW.get("18:24", {}).get("inputs", {}).get("switch", False)),
    "lora1": "None",
    "lora1_strength": 1.0,
    "lora2": "None",
    "lora2_strength": 1.0,
}

app = Flask(__name__)

def comfy_get(path, timeout=15, **kw):
    r = requests.get(COMFY + path, timeout=timeout, **kw)
    r.raise_for_status()
    return r.json()

def combo(spec):
    if not spec:
        return []
    if isinstance(spec[0], list):
        return spec[0]
    if len(spec) > 1 and isinstance(spec[1], dict) and "options" in spec[1]:
        return list(spec[1]["options"])
    return []

def required_inputs(node_class):
    try:
        return comfy_get(f"/object_info/{node_class}")[node_class]["input"]["required"]
    except Exception:
        return {}

def clamp(value, lo, hi, cast, default):
    try:
        return max(lo, min(hi, cast(value)))
    except (TypeError, ValueError):
        return default

def build_workflow(p):
    wf = copy.deepcopy(WORKFLOW)
    seed = clamp(p.get("seed", -1), -1, 2**50, int, -1)
    if seed < 0:
        seed = random.randint(0, 2**50)

    wf["18:17"]["inputs"]["value"] = (p.get("prompt") or "").strip()
    wf["18:5"]["inputs"]["text"] = (p.get("negative") or "").strip()

    ks = wf["18:7"]["inputs"]
    ks["seed"] = seed
    ks["steps"] = clamp(p.get("steps"), 1, 150, int, DEFAULTS["steps"])
    ks["cfg"] = clamp(p.get("cfg"), 0, 30, float, DEFAULTS["cfg"])
    ks["sampler_name"] = p.get("sampler") or DEFAULTS["sampler"]
    ks["scheduler"] = p.get("scheduler") or DEFAULTS["scheduler"]

    wf["19"]["inputs"]["aspect_ratio"] = p.get("aspect_ratio") or DEFAULTS["aspect_ratio"]
    wf["19"]["inputs"]["megapixels"] = clamp(p.get("megapixels"), 0.25, 4, float, 1.0)
    wf["18:16"]["inputs"]["switch"] = bool(p.get("upsampler", DEFAULTS["upsampler"]))
    wf["18:24"]["inputs"]["switch"] = bool(p.get("slg", DEFAULTS["slg"]))
    wf["18:11"]["inputs"]["sampling_mode.seed"] = seed
    wf["18:1"]["inputs"]["unet_name"] = p.get("checkpoint") or DEFAULTS["checkpoint"]

    curr_model = ["18:1", 0]
    curr_clip = ["18:2", 0]

    l1_name = p.get("lora1", "None")
    l1_str = float(p.get("lora1_strength", 1.0))
    # Turbo LoRA Protection: 8-step DMD LoRA requires CFG <= 1.5 and SLG off to avoid NaN latent explosion
    if "8-step" in (str(l1_name) + str(p.get("lora2", ""))).lower():
        if ks["cfg"] > 1.5:
            ks["cfg"] = 1.0
        wf["18:24"]["inputs"]["switch"] = False
    if l1_name and l1_name != "None" and l1_str != 0:
        wf["lora_1"] = {
            "inputs": {"lora_name": l1_name, "strength_model": l1_str, "strength_clip": l1_str, "model": curr_model, "clip": curr_clip},
            "class_type": "LoraLoader",
            "_meta": {"title": f"LoRA 1 ({l1_name})"}
        }
        curr_model = ["lora_1", 0]
        curr_clip = ["lora_1", 1]

    l2_name = p.get("lora2", "None")
    l2_str = float(p.get("lora2_strength", 1.0))
    if l2_name and l2_name != "None" and l2_str != 0:
        wf["lora_2"] = {
            "inputs": {"lora_name": l2_name, "strength_model": l2_str, "strength_clip": l2_str, "model": curr_model, "clip": curr_clip},
            "class_type": "LoraLoader",
            "_meta": {"title": f"LoRA 2 ({l2_name})"}
        }
        curr_model = ["lora_2", 0]
        curr_clip = ["lora_2", 1]

    if curr_model != ["18:1", 0]:
        wf["18:15"]["inputs"]["model"] = curr_model
        wf["18:24"]["inputs"]["on_false"] = curr_model
    if curr_clip != ["18:2", 0]:
        wf["18:4"]["inputs"]["clip"] = curr_clip
        wf["18:5"]["inputs"]["clip"] = curr_clip
        if "18:11" in wf and "clip" in wf["18:11"]["inputs"]:
            wf["18:11"]["inputs"]["clip"] = curr_clip

    return wf, seed

def explain_validation(resp):
    try:
        data = resp.json()
    except ValueError:
        return f"Backend rejected the request ({resp.status_code})."
    parts = []
    err = data.get("error")
    if isinstance(err, dict):
        parts.append(err.get("message", "Invalid workflow"))
        if err.get("details"):
            parts.append(str(err["details"]))
    for nid, info in (data.get("node_errors") or {}).items():
        for e in info.get("errors", []):
            parts.append(f'{info.get("class_type", nid)}: {e.get("message", "")} {e.get("details", "")}'.strip())
    return " | ".join(parts) or "Backend rejected the workflow."

def find_error(entry):
    for kind, data in entry.get("status", {}).get("messages", []):
        if kind == "execution_error" and isinstance(data, dict):
            return f'{data.get("node_type", "node")}: {data.get("exception_message", "execution failed")}'.strip()
    return None

@app.get("/")
def index():
    return Response(PAGE, mimetype="text/html")

@app.get("/api/health")
def health():
    try:
        comfy_get("/system_stats", timeout=3)
        return jsonify(ok=True)
    except Exception:
        return jsonify(ok=False), 503

@app.get("/api/options")
def options():
    res = required_inputs("ResolutionSelector")
    ks = required_inputs("KSampler")
    un = required_inputs("UNETLoader")
    checkpoints = combo(un.get("unet_name"))
    for std_ckpt in ["moss_image_sft_int8_convrot.safetensors", "moss_image_base_int8_convrot.safetensors", "kromoss_int8_convrot.safetensors"]:
        if std_ckpt not in checkpoints:
            checkpoints.append(std_ckpt)

    lora_spec = required_inputs("LoraLoader")
    loras = ["None"] + [l for l in combo(lora_spec.get("lora_name")) if l != "None"]
    lora_dir = os.path.join(COMFY_DIR, "models", "loras")
    if os.path.exists(lora_dir):
        for root, _, files in os.walk(lora_dir):
            for f in files:
                if f.endswith((".safetensors", ".pt", ".bin", ".ckpt")):
                    rel = os.path.relpath(os.path.join(root, f), lora_dir)
                    if rel not in loras:
                        loras.append(rel)

    return jsonify(
        aspect_ratios=combo(res.get("aspect_ratio")) or [DEFAULTS["aspect_ratio"]],
        samplers=combo(ks.get("sampler_name")) or [DEFAULTS["sampler"]],
        schedulers=combo(ks.get("scheduler")) or [DEFAULTS["scheduler"]],
        checkpoints=checkpoints,
        loras=loras,
        defaults=DEFAULTS,
    )

@app.post("/api/generate")
def generate():
    p = request.get_json(force=True, silent=True) or {}
    if not (p.get("prompt") or "").strip():
        return jsonify(error="Please enter a prompt."), 400
    wf, seed = build_workflow(p)
    try:
        r = requests.post(
            f"{COMFY}/prompt",
            json={
                "prompt": wf,
                "client_id": CLIENT_ID,
                "extra_data": {"extra_pnginfo": {"workflow": wf}}
            },
            timeout=30
        )
    except requests.RequestException as e:
        return jsonify(error=f"Backend is not reachable: {e}"), 502
    if r.status_code != 200:
        return jsonify(error=explain_validation(r)), 400
    prompt_id = r.json()["prompt_id"]
    JOB_PARAMS[prompt_id] = {"params": p, "seed": seed, "workflow": wf}
    return jsonify(prompt_id=prompt_id, seed=seed)

@app.get("/api/status/<pid>")
def status(pid):
    try:
        hist = comfy_get(f"/history/{pid}")
    except Exception as e:
        return jsonify(state="error", error=f"Backend is not reachable: {e}"), 502

    if pid in hist:
        entry = hist[pid]
        outputs = entry.get("outputs", {})
        images = outputs.get("9", {}).get("images", [])
        if images:
            caption = ""
            preview = outputs.get("18:13", {}).get("text")
            if preview:
                caption = preview[0] if isinstance(preview, list) else str(preview)
            meta_info = JOB_PARAMS.get(pid, {})
            for img in images:
                subfolder = img.get("subfolder", "")
                filename = img.get("filename", "")
                out_dir = os.path.join(COMFY_DIR, "output", subfolder) if subfolder else os.path.join(COMFY_DIR, "output")
                img_path = os.path.join(out_dir, filename)
                embed_png_metadata(img_path, meta_info, caption)
            return jsonify(state="done", images=images, caption=caption)
        err = find_error(entry)
        if err:
            return jsonify(state="error", error=err)
        if entry.get("status", {}).get("completed"):
            return jsonify(state="error", error="The workflow finished but produced no image.")

    try:
        q = comfy_get("/queue")
    except Exception:
        return jsonify(state="queued", position=None)
    if any(len(i) > 1 and i[1] == pid for i in q.get("queue_running", [])):
        return jsonify(state="running")
    pending = [i[1] for i in q.get("queue_pending", []) if len(i) > 1]
    if pid in pending:
        return jsonify(state="queued", position=pending.index(pid) + 1)
    return jsonify(state="running")

@app.post("/api/cancel")
def cancel():
    pid = (request.get_json(silent=True) or {}).get("prompt_id")
    if not pid:
        return jsonify(ok=False), 400
    try:
        q = comfy_get("/queue")
        if any(len(i) > 1 and i[1] == pid for i in q.get("queue_running", [])):
            requests.post(f"{COMFY}/interrupt", timeout=10)
        else:
            requests.post(f"{COMFY}/queue", json={"delete": [pid]}, timeout=10)
    except Exception as e:
        return jsonify(ok=False, error=str(e)), 502
    return jsonify(ok=True)

@app.get("/api/image")
def image():
    params = {"filename": request.args.get("filename", ""), "subfolder": request.args.get("subfolder", ""), "type": request.args.get("type", "output")}
    if params["type"] not in ("output", "temp") or not params["filename"]:
        return "bad request", 400
    subfolder = params["subfolder"]
    filename = params["filename"]
    out_dir = os.path.join(COMFY_DIR, "output", subfolder) if subfolder else os.path.join(COMFY_DIR, "output")
    local_path = os.path.join(out_dir, filename)
    if os.path.isfile(local_path):
        with open(local_path, "rb") as f:
            content = f.read()
        return Response(content, mimetype="image/png", headers={
            "Cache-Control": "public, max-age=3600",
            "Content-Disposition": f'inline; filename="{filename}"'
        })
    r = requests.get(f"{COMFY}/view", params=params, timeout=60)
    if r.status_code != 200:
        return "not found", 404
    return Response(r.content, mimetype=r.headers.get("Content-Type", "image/png"), headers={"Cache-Control": "public, max-age=3600"})

PAGE = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>MOSS-Image & Kromoss</title>
<style>
:root{--bg:#0e0f13;--panel:#171922;--line:#262a38;--txt:#e8eaf0;--mut:#8b91a5;--acc:#7c8cff;--acc2:#a78bfa;--err:#ff6b6b}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--txt);font:15px/1.45 system-ui,-apple-system,sans-serif}
header{display:flex;align-items:center;gap:12px;padding:14px 20px;border-bottom:1px solid var(--line)}
header h1{font-size:18px;margin:0;font-weight:650}#conn{margin-left:auto;font-size:12px;color:var(--mut)}
#conn i{display:inline-block;width:8px;height:8px;border-radius:50%;background:#555;margin-right:6px}
#conn.ok i{background:#3ddc97}#conn.bad i{background:var(--err)}
main{display:grid;grid-template-columns:minmax(320px,460px) 1fr;gap:16px;padding:16px;max-width:1400px;margin:0 auto}
@media(max-width:860px){main{grid-template-columns:1fr}}
.panel{background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:16px}
label{display:block;font-size:12px;color:var(--mut);margin:12px 0 5px;text-transform:uppercase;letter-spacing:.5px}
label:first-child{margin-top:0}
textarea,input[type=number],select{width:100%;background:#0f1117;color:var(--txt);border:1px solid var(--line);border-radius:9px;padding:9px 10px;font:inherit}
textarea{resize:vertical}input[type=range]{width:100%;accent-color:var(--acc)}
.row{display:grid;grid-template-columns:1fr 1fr;gap:10px}
.chk{display:flex;gap:9px;align-items:flex-start;font-size:14px;margin:12px 0 0}.chk input{margin-top:3px;accent-color:var(--acc)}
.chk small{display:block;color:var(--mut);font-size:12px}
.lora-box{background:#11131a;border:1px solid var(--line);border-radius:10px;padding:10px;margin-top:12px}
.btns{display:flex;gap:10px;margin-top:16px}
button{border:0;border-radius:10px;padding:11px 16px;font:inherit;font-weight:600;cursor:pointer;color:#fff;background:linear-gradient(135deg,var(--acc),var(--acc2))}
button:disabled{opacity:.5;cursor:not-allowed}button.ghost{background:transparent;border:1px solid var(--line);color:var(--txt)}
#go{flex:1}
#stage{min-height:300px;display:flex;align-items:center;justify-content:center;background:#0f1117;border:1px dashed var(--line);border-radius:12px;overflow:hidden}
#stage img{max-width:100%;max-height:78vh;display:block}#placeholder{color:var(--mut)}#status{margin:12px 2px 0;color:var(--mut);min-height:20px}
#status.err{color:var(--err)}#actions{display:flex;gap:10px;margin-top:10px;align-items:center;flex-wrap:wrap}#actions a{color:var(--acc);text-decoration:none;font-weight:600}
details{margin-top:12px;background:#0f1117;border:1px solid var(--line);border-radius:10px;padding:8px 12px}summary{cursor:pointer;color:var(--mut)}
pre{white-space:pre-wrap;word-break:break-word;margin:10px 0 2px;font:13px/1.5 ui-monospace,Menlo,monospace}
h3{font-size:13px;color:var(--mut);text-transform:uppercase;letter-spacing:.5px;margin:20px 0 8px}
#hist{display:grid;grid-template-columns:repeat(auto-fill,minmax(96px,1fr));gap:8px}
#hist img{width:100%;aspect-ratio:1;object-fit:cover;border-radius:8px;border:1px solid var(--line);cursor:pointer}[hidden]{display:none!important}
</style>
</head>
<body>
<header><h1>MOSS-Image & Kromoss</h1><span id="conn"><i></i>connecting…</span></header>
<main>
  <section class="panel">
    <label for="prompt">Prompt</label><textarea id="prompt" rows="6"></textarea>
    <label for="negative">Negative prompt</label><textarea id="negative" rows="2"></textarea>
    <div class="row">
      <div><label for="aspect">Aspect ratio</label><select id="aspect"></select></div>
      <div><label for="ckpt">Checkpoint</label><select id="ckpt"></select></div>
    </div>
    <div class="lora-box">
      <div class="row">
        <div><label for="lora1">LoRA 1</label><select id="lora1"></select></div>
        <div><label for="lora1_str">LoRA 1 Strength</label><input type="number" id="lora1_str" min="-2" max="2" step="0.05" value="1.0"></div>
      </div>
      <div class="row" style="margin-top:6px">
        <div><label for="lora2">LoRA 2</label><select id="lora2"></select></div>
        <div><label for="lora2_str">LoRA 2 Strength</label><input type="number" id="lora2_str" min="-2" max="2" step="0.05" value="1.0"></div>
      </div>
    </div>
    <label for="mp">Megapixels: <b id="mpv"></b></label><input type="range" id="mp" min="0.25" max="4" step="0.05">
    <div class="row">
      <div><label for="steps">Steps</label><input type="number" id="steps" min="1" max="150"></div>
      <div><label for="cfg">CFG</label><input type="number" id="cfg" min="0" max="30" step="0.1"></div>
    </div>
    <div class="row">
      <div><label for="sampler">Sampler</label><select id="sampler"></select></div>
      <div><label for="scheduler">Scheduler</label><select id="scheduler"></select></div>
    </div>
    <div class="row">
      <div><label for="seed">Seed</label><input type="number" id="seed" min="0"></div>
      <label class="chk" style="margin-top:28px"><input type="checkbox" id="rand" checked><span>Random seed</span></label>
    </div>
    <label class="chk"><input type="checkbox" id="upsampler"><span>Prompt upsampler<small>Expands concise ideas into comprehensive scene captions</small></span></label>
    <label class="chk"><input type="checkbox" id="slg"><span>Skip-layer guidance<small>Layer 22 guidance for enhanced details</small></span></label>
    <div class="btns"><button id="go">Generate</button><button id="stop" class="ghost" hidden>Cancel</button></div>
  </section>
  <section class="panel">
    <div id="stage"><span id="placeholder">Your image will appear here</span></div>
    <div id="status"></div>
    <div id="actions" hidden><a id="dl" download>Download</a><button id="reuse" class="ghost">Reuse this seed</button></div>
    <details id="capt" hidden><summary>Final prompt sent to the model</summary><pre id="captxt"></pre></details>
    <h3>History</h3><div id="hist"></div>
  </section>
</main>
<script>
const $ = s => document.querySelector(s);
const HKEY = "moss_history_v1";
let pid = null, tStart = 0, polling = false, history = [], lastSeed = null;
function fill(sel, items, value){
  const el = $(sel); el.innerHTML = "";
  items.forEach(v => { const o = document.createElement("option"); o.value = o.textContent = v; el.appendChild(o); });
  if (value !== undefined) el.value = value;
}
function setSettings(s){
  $("#prompt").value = s.prompt; $("#negative").value = s.negative;
  $("#aspect").value = s.aspect_ratio; $("#ckpt").value = s.checkpoint;
  $("#lora1").value = s.lora1 || "None"; $("#lora1_str").value = s.lora1_strength ?? 1.0;
  $("#lora2").value = s.lora2 || "None"; $("#lora2_str").value = s.lora2_strength ?? 1.0;
  $("#mp").value = s.megapixels; $("#mpv").textContent = (+s.megapixels).toFixed(2);
  $("#steps").value = s.steps; $("#cfg").value = s.cfg;
  $("#sampler").value = s.sampler; $("#scheduler").value = s.scheduler;
  $("#upsampler").checked = !!s.upsampler; $("#slg").checked = !!s.slg;
}
function getSettings(){
  return {
    prompt: $("#prompt").value, negative: $("#negative").value,
    aspect_ratio: $("#aspect").value, checkpoint: $("#ckpt").value,
    lora1: $("#lora1").value, lora1_strength: parseFloat($("#lora1_str").value) || 0,
    lora2: $("#lora2").value, lora2_strength: parseFloat($("#lora2_str").value) || 0,
    megapixels: parseFloat($("#mp").value), steps: parseInt($("#steps").value, 10),
    cfg: parseFloat($("#cfg").value), sampler: $("#sampler").value, scheduler: $("#scheduler").value,
    seed: $("#rand").checked ? -1 : parseInt($("#seed").value || "0", 10),
    upsampler: $("#upsampler").checked, slg: $("#slg").checked
  };
}
function say(msg, isErr){ const s = $("#status"); s.textContent = msg; s.className = isErr ? "err" : ""; }
function busy(b){ $("#go").disabled = b; $("#stop").hidden = !b; }
function imgUrl(i){
  return "/api/image?filename=" + encodeURIComponent(i.filename) + "&subfolder=" + encodeURIComponent(i.subfolder || "") + "&type=" + encodeURIComponent(i.type || "output");
}
function show(url, caption, seed, filename){
  const img = new Image();
  img.onload = () => { say("Done \u2022 " + img.naturalWidth + "\u00d7" + img.naturalHeight + " \u2022 seed " + seed); };
  img.src = url; $("#stage").innerHTML = ""; $("#stage").appendChild(img);
  $("#dl").href = url;
  if (filename) $("#dl").setAttribute("download", filename);
  $("#actions").hidden = false;
  $("#captxt").textContent = caption || ""; $("#capt").hidden = !caption; lastSeed = seed;
}
function saveHistory(){ try { localStorage.setItem(HKEY, JSON.stringify(history.slice(0, 24))); } catch(e){} }
function loadHistory(){ try { history = JSON.parse(localStorage.getItem(HKEY) || "[]"); } catch(e){ history = []; } }
function renderHistory(){
  const box = $("#hist"); box.innerHTML = "";
  history.forEach(h => {
    const t = new Image(); t.src = h.url; t.loading = "lazy"; t.title = h.settings.prompt.slice(0, 120);
    t.onerror = () => t.remove();
    t.onclick = () => { setSettings(h.settings); $("#rand").checked = false; $("#seed").value = h.settings.seed; show(h.url, h.caption, h.settings.seed, h.filename); };
    box.appendChild(t);
  });
}
async function poll(id){
  if (polling) return;
  polling = true;
  let fails = 0;
  try {
    while (pid === id) {
      try {
        const r = await fetch("/api/status/" + id);
        if (!r.ok) throw new Error("HTTP " + r.status);
        const d = await r.json();
        fails = 0;
        const secs = Math.round((Date.now() - tStart) / 1000);
        if (d.state === "done") {
          const url = imgUrl(d.images[0]); const s = Object.assign({}, window._lastSettings);
          const fname = (d.images && d.images[0] && d.images[0].filename) ? d.images[0].filename : "";
          show(url, d.caption, s.seed, fname);
          history.unshift({url, caption: d.caption, settings: s, filename: fname}); saveHistory(); renderHistory();
          pid = null; busy(false); return;
        }
        if (d.state === "error") { say(d.error || "Generation failed.", true); pid = null; busy(false); return; }
        if (d.state === "queued") say("Queued" + (d.position ? " (position " + d.position + ")" : "") + " \u2022 " + secs + "s");
        else say("Generating\u2026 " + secs + "s  (first run loads models and takes longer)");
      } catch (err) {
        fails++;
        if (fails >= 6) { throw err; }
        say("Tunnel reconnecting (" + fails + "/6)\u2026 model inference still running");
      }
      await new Promise(r => setTimeout(r, 2500));
    }
  } catch (e) { say("Lost connection to the server: " + e, true); pid = null; busy(false); } finally { polling = false; }
}
async function generate(){
  const s = getSettings();
  if (!s.prompt.trim()) { say("Please enter a prompt.", true); return; }
  busy(true); say("Submitting\u2026");
  try {
    const r = await fetch("/api/generate", {method: "POST", headers: {"Content-Type": "application/json"}, body: JSON.stringify(s)});
    const d = await r.json();
    if (!r.ok) { say(d.error || "Request failed.", true); busy(false); return; }
    s.seed = d.seed; window._lastSettings = s;
    if ($("#rand").checked) $("#seed").value = d.seed;
    pid = d.prompt_id; tStart = Date.now(); poll(pid);
  } catch (e) { say("Could not reach the server: " + e, true); busy(false); }
}
async function cancel(){
  if (!pid) return; const id = pid; pid = null;
  try { await fetch("/api/cancel", {method: "POST", headers: {"Content-Type": "application/json"}, body: JSON.stringify({prompt_id: id})}); } catch(e){}
  busy(false); say("Cancelled.");
}
async function health(){
  try {
    const r = await fetch("/api/health"); const el = $("#conn");
    el.className = r.ok ? "ok" : "bad"; el.innerHTML = "<i></i>" + (r.ok ? "Backend online" : "Backend offline");
  } catch(e){ const el = $("#conn"); el.className = "bad"; el.innerHTML = "<i></i>server unreachable"; }
}
async function init(){
  loadHistory();
  const o = await (await fetch("/api/options")).json();
  const d = o.defaults;
  fill("#aspect", o.aspect_ratios, d.aspect_ratio); fill("#ckpt", o.checkpoints, d.checkpoint);
  fill("#lora1", o.loras, d.lora1); fill("#lora2", o.loras, d.lora2);
  $("#lora1").addEventListener("change", e => {
    if (e.target.value && e.target.value.toLowerCase().includes("8-step")) {
      $("#steps").value = 8;
      $("#cfg").value = 1.0;
      $("#slg").checked = false;
      say("⚡ Turbo LoRA active: Steps set to 8, CFG to 1.0, SLG turned off.");
    }
  });
  fill("#sampler", o.samplers, d.sampler); fill("#scheduler", o.schedulers, d.scheduler);
  setSettings(d);
  renderHistory(); health(); setInterval(health, 15000);
}
$("#mp").addEventListener("input", e => $("#mpv").textContent = (+e.target.value).toFixed(2));
$("#go").onclick = generate; $("#stop").onclick = cancel;
$("#reuse").onclick = () => { if (lastSeed !== null) { $("#rand").checked = false; $("#seed").value = lastSeed; } };
$("#prompt").addEventListener("keydown", e => { if ((e.ctrlKey || e.metaKey) && e.key === "Enter") generate(); });
init();
</script>
</body>
</html>"""

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=int(os.environ.get("PORT", 5000)), threaded=True)
'''

open(f"{APP_DIR}/app.py", "w", encoding="utf-8").write(APP_CODE)
print("✅ Setup completed")

In [ ]:
#@title 2️⃣ Download Model Checkpoints & LoRAs { display-mode: "form" }
DOWNLOAD_TURBO_LORA = True  #@param {type:"boolean"}
DOWNLOAD_KROMOSS = False  #@param {type:"boolean"}
DOWNLOAD_BASE_CHECKPOINT = True  #@param {type:"boolean"}
EXTRA_LORA_URLS = ""  #@param {type:"string"}
EXTRA_MODEL_URLS = ""  #@param {type:"string"}
CIVITAI_API_KEY = ""  #@param {type:"string"}

import os, re, subprocess, sys, urllib.parse
import requests

COMFY_DIR = "/content/ComfyUI"
LORA_DIR = f"{COMFY_DIR}/models/loras"
MODEL_DIR = f"{COMFY_DIR}/models/diffusion_models"
HF_MOSS = "https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main"
HF_KROMOSS = "https://huggingface.co/lodestones/Kromoss/resolve/main"

os.makedirs(LORA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

DOWNLOADS = [
    ("diffusion_models", "moss_image_sft_int8_convrot.safetensors", f"{HF_MOSS}/diffusion_models/moss_image_sft_int8_convrot.safetensors"),
    ("text_encoders",    "qwen3.5_9b_w4a8_gptq.safetensors",       f"{HF_MOSS}/text_encoders/qwen3.5_9b_w4a8_gptq.safetensors"),
    ("vae",              "flux2-vae_small.safetensors",             f"{HF_MOSS}/vae/flux2-vae_small.safetensors"),
]

if DOWNLOAD_BASE_CHECKPOINT:
    DOWNLOADS.append(("diffusion_models", "moss_image_base_int8_convrot.safetensors", f"{HF_MOSS}/diffusion_models/moss_image_base_int8_convrot.safetensors"))

if DOWNLOAD_KROMOSS:
    DOWNLOADS.append(("diffusion_models", "kromoss_int8_convrot.safetensors", f"{HF_KROMOSS}/diffusion_models/kromoss_int8_convrot.safetensors"))

if DOWNLOAD_TURBO_LORA:
    DOWNLOADS.append(("loras", "8-steps-dmd-lora.safetensors", f"{HF_KROMOSS}/8-steps-dmd-lora.safetensors"))

def aria2_download(url, dest_dir, filename):
    subprocess.run([
        "aria2c", "-x", "16", "-s", "16", "-k", "1M", "-c", "--file-allocation=none",
        "--console-log-level=warn", "--summary-interval=10",
        "-d", dest_dir, "-o", filename, url,
    ], check=True)

for folder, name, url in DOWNLOADS:
    dest = f"{COMFY_DIR}/models/{folder}"
    os.makedirs(dest, exist_ok=True)
    path = f"{dest}/{name}"
    if os.path.exists(path) and not os.path.exists(path + ".aria2") and os.path.getsize(path) > 10_000_000:
        print(f"✓ {name} already downloaded")
        continue
    print(f"⬇️  {name}")
    aria2_download(url, dest, name)

VALID_EXTS = (".safetensors", ".pt", ".bin", ".ckpt")

def ensure_valid_ext(fname, default_ext=".safetensors"):
    fname = re.sub(r'[<>:"/\\|?*]', "_", fname.strip())
    if not fname.lower().endswith(VALID_EXTS):
        fname += default_ext
    return fname

def filename_from_headers(headers, final_url=""):
    cd = headers.get("Content-Disposition", "")
    if cd:
        m = re.search(r"filename\*=UTF-8''([^;]+)", cd, re.I)
        if m:
            return urllib.parse.unquote(m.group(1).strip("\"' "))
        m = re.search(r'filename="?([^";]+)"?', cd, re.I)
        if m:
            return urllib.parse.unquote(m.group(1).strip("\"' "))
    if final_url:
        qs = urllib.parse.parse_qs(urllib.parse.urlparse(final_url).query)
        rcd = qs.get("response-content-disposition", [""])[0]
        if rcd:
            m = re.search(r'filename="?([^";]+)"?', urllib.parse.unquote(rcd), re.I)
            if m:
                return m.group(1).strip("\"' ")
    return None

def download_from_gdrive(url, dest_dir, custom_name=None, label="LoRA"):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "gdown"], check=True)
        import gdown

    if "/folders/" in url:
        print(f"⬇️ Extra {label} (Google Drive Folder): {url}")
        gdown.download_folder(url=url, output=dest_dir, quiet=False, remaining_ok=True)
        return

    print(f"⬇️ Extra {label} (Google Drive): {url}")
    out_target = os.path.join(dest_dir, ensure_valid_ext(custom_name)) if custom_name else (dest_dir + "/")
    out_path = gdown.download(url=url, output=out_target, quiet=False, fuzzy=True)
    if not out_path or not os.path.exists(out_path):
        raise RuntimeError(f"Google Drive download failed for: {url}. Ensure file sharing is set to 'Anyone with the link'.")
    if not out_path.lower().endswith(VALID_EXTS):
        new_path = out_path + ".safetensors"
        os.replace(out_path, new_path)
        out_path = new_path
    print(f"✓ Saved Google Drive {label}: {os.path.basename(out_path)}")

def download_from_civitai(url, dest_dir, api_key="", custom_name=None, label="LoRA"):
    parsed = urllib.parse.urlparse(url)
    qs = urllib.parse.parse_qs(parsed.query)
    headers = {"User-Agent": "Mozilla/5.0"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    version_id = None
    dl_url = url
    fname = custom_name

    # Case 1: Model page URL (e.g., https://civitai.com/models/12345 or ?modelVersionId=67890)
    m_page = re.search(r"/models/(\d+)", parsed.path)
    m_dl = re.search(r"/api/download/models/(\d+)", parsed.path)

    if m_dl:
        version_id = m_dl.group(1)
    elif m_page:
        if "modelVersionId" in qs:
            version_id = qs["modelVersionId"][0]
        else:
            model_id = m_page.group(1)
            r = requests.get(f"https://civitai.com/api/v1/models/{model_id}", headers=headers, timeout=30)
            r.raise_for_status()
            versions = r.json().get("modelVersions") or []
            if not versions:
                raise RuntimeError(f"No model versions found on Civitai for model {model_id}")
            version_id = str(versions[0]["id"])
        dl_url = f"https://civitai.com/api/download/models/{version_id}"

    # Fetch filename & download URL from Civitai API if version_id is known
    if version_id and not fname:
        try:
            r_ver = requests.get(f"https://civitai.com/api/v1/model-versions/{version_id}", headers=headers, timeout=30)
            if r_ver.status_code == 200:
                vdata = r_ver.json()
                files = vdata.get("files") or []
                chosen = next((f for f in files if f.get("primary")), files[0] if files else None)
                if chosen:
                    fname = chosen.get("name")
                    if chosen.get("downloadUrl"):
                        dl_url = chosen["downloadUrl"]
        except Exception:
            pass

    # Append token query parameter if provided
    if api_key and "token=" not in dl_url:
        sep = "&" if "?" in dl_url else "?"
        dl_url = f"{dl_url}{sep}token={urllib.parse.quote(api_key)}"

    # Resolve redirect to get the direct signed CDN URL and Content-Disposition filename
    resp = requests.get(dl_url, headers={"User-Agent": "Mozilla/5.0"}, allow_redirects=False, timeout=30)
    if resp.status_code in (301, 302, 303, 307, 308) and "Location" in resp.headers:
        final_url = resp.headers["Location"]
        if "civitai.com/login" in final_url:
            raise RuntimeError(
                f"Civitai requires authentication to download this {label} ({url}). "
                "Please paste your Civitai API Key into CIVITAI_API_KEY and re-run."
            )
        if not fname:
            fname = filename_from_headers(resp.headers, final_url)
    elif resp.status_code in (401, 403):
        raise RuntimeError(
            f"Civitai returned HTTP {resp.status_code} for {url}. "
            "Please provide a valid CIVITAI_API_KEY in the form above."
        )
    else:
        resp.raise_for_status()
        final_url = dl_url
        if not fname:
            fname = filename_from_headers(resp.headers, final_url)

    if not fname:
        fname = f"civitai_{label.lower()}_{version_id or 'custom'}.safetensors"
    fname = ensure_valid_ext(fname)

    print(f"⬇️ Extra {label} (Civitai): {fname}")
    aria2_download(final_url, dest_dir, fname)

def download_extra(entry, dest_dir, civitai_key="", label="LoRA"):
    custom_name = None
    url = entry.strip()
    # Support optional custom_name.safetensors|https://... syntax
    if "|" in url and not url.startswith(("http://", "https://")):
        custom_name, url = [p.strip() for p in url.split("|", 1)]

    domain = urllib.parse.urlparse(url).netloc.lower()

    if "drive.google.com" in domain or "docs.google.com" in domain:
        download_from_gdrive(url, dest_dir, custom_name, label)
    elif "civitai.com" in domain:
        download_from_civitai(url, dest_dir, civitai_key, custom_name, label)
    else:
        # Auto-fix Hugging Face /blob/ links to /resolve/
        if "huggingface.co" in domain and "/blob/" in url:
            url = url.replace("/blob/", "/resolve/", 1)
        fname = custom_name or urllib.parse.unquote(url.split("?")[0].rstrip("/").split("/")[-1])
        fname = ensure_valid_ext(fname)
        print(f"⬇️ Extra {label}: {fname}")
        aria2_download(url, dest_dir, fname)

civitai_key = CIVITAI_API_KEY.strip()
for raw_entry in EXTRA_MODEL_URLS.replace(",", " ").split():
    download_extra(raw_entry, MODEL_DIR, civitai_key, "Model")
for raw_entry in EXTRA_LORA_URLS.replace(",", " ").split():
    download_extra(raw_entry, LORA_DIR, civitai_key, "LoRA")

print("✅ Checkpoints & LoRAs ready")

In [ ]:
#@title 🚀 3️⃣ Launch WebUI via Public URL { display-mode: "form" }
PINGGY_TOKEN = ""  #@param {type:"string"}
EXTRA_ARGS = ""  #@param {type:"string"}

import json, os, re, shlex, subprocess, sys, time
import requests
from IPython.display import HTML, display

COMFY_DIR, APP_DIR, LOG_DIR = "/content/ComfyUI", "/content/moss_ui", "/content/logs"
COMFY_PORT, APP_PORT = 8188, 5000
os.makedirs(LOG_DIR, exist_ok=True)
COMFY_LOG, APP_LOG, TUNNEL_LOG = (f"{LOG_DIR}/{n}.log" for n in ("backend", "flask", "pinggy"))

uv_bin_dir = os.path.expanduser("~/.local/bin")
if uv_bin_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{uv_bin_dir}:{os.environ.get('PATH', '')}"

subprocess.run("pkill -f '[C]omfyUI/main.py'; pkill -f '[m]oss_ui/app.py'; pkill -f '[a].pinggy.io'", shell=True)
time.sleep(1)

env = dict(
    os.environ,
    PYTHONUNBUFFERED="1",
    PYTORCH_CUDA_ALLOC_CONF="backend:cudaMallocAsync",
    COMFY_KITCHEN_DISABLE_CUTLASS="1",
)

def wait_http(url, proc, log, timeout, name):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            print(open(log, errors="replace").read()[-3000:])
            raise RuntimeError(f"{name} exited, see the log above.")
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return
        except requests.RequestException:
            pass
        time.sleep(1)
    raise TimeoutError(f"{name} did not start within {timeout}s")

# 1) Start generation engine
print("Starting generation engine…")
backend = subprocess.Popen(
    [sys.executable, "-u", f"{COMFY_DIR}/main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT),
     "--disable-auto-launch", *shlex.split(EXTRA_ARGS)],
    cwd=COMFY_DIR, stdout=open(COMFY_LOG, "w"), stderr=subprocess.STDOUT, env=env)
wait_http(f"http://127.0.0.1:{COMFY_PORT}/system_stats", backend, COMFY_LOG, 300, "Generation Engine")

needed = {n["class_type"] for n in json.load(open(f"{APP_DIR}/workflow_api.json")).values()}
known = requests.get(f"http://127.0.0.1:{COMFY_PORT}/object_info", timeout=60).json()
missing = sorted(needed - set(known))
if missing:
    raise RuntimeError(f"Missing workflow nodes {missing}. Re-run cell 1 to update.")
print("✓ Generation engine ready")

# 2) Start web interface
flask_proc = subprocess.Popen(
    [sys.executable, "-u", f"{APP_DIR}/app.py"], cwd=APP_DIR,
    stdout=open(APP_LOG, "w"), stderr=subprocess.STDOUT,
    env=dict(env, PORT=str(APP_PORT), COMFY_URL=f"http://127.0.0.1:{COMFY_PORT}"))
wait_http(f"http://127.0.0.1:{APP_PORT}/api/health", flask_proc, APP_LOG, 30, "Web UI")
print("✓ Web UI ready")

# 3) Start public access tunnel
ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]|\x1b\([A-Za-z]|\r")
URL_RE = re.compile(r"https://[A-Za-z0-9.-]+\.(?:(?:free\.)?pinggy(?:-free)?\.link|pinggy\.online)")
STRIP_CHARS = ".,;')\""

def extract_tunnel_url(text):
    clean = ANSI.sub("", text)
    m = URL_RE.search(clean)
    if m:
        return m.group(0).rstrip(STRIP_CHARS)
    for raw in re.findall(r"https://\S+", clean):
        candidate = raw.rstrip(STRIP_CHARS)
        if "pinggy" in candidate and not candidate.startswith(("https://pinggy.io", "https://admin.pinggy.io")):
            return candidate
    return None

def start_tunnel():
    user = (PINGGY_TOKEN.strip() + "@") if PINGGY_TOKEN.strip() else "free@"
    host = f"{user}a.pinggy.io"
    cmd = [
        "ssh", "-p", "443", f"-R0:localhost:{APP_PORT}",
        "-o", "StrictHostKeyChecking=no",
        "-o", "UserKnownHostsFile=/dev/null",
        "-o", "ServerAliveInterval=15",
        "-o", "ServerAliveCountMax=5",
        "-o", "TCPKeepAlive=yes",
        "-o", "ExitOnForwardFailure=yes",
        host
    ]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=open(TUNNEL_LOG, "w"), stderr=subprocess.STDOUT)
    t0 = time.time()
    while time.time() - t0 < 45:
        if proc.poll() is not None:
            break
        found = extract_tunnel_url(open(TUNNEL_LOG, errors="replace").read())
        if found:
            return proc, found
        time.sleep(1)
    print(open(TUNNEL_LOG, errors="replace").read()[-1500:])
    raise RuntimeError("Public tunnel did not come up.")

def announce(url):
    print(f"\n🌍 Public URL: {url}\n")
    display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px;font-weight:600">Open MOSS-Image Web App ↗</a>'))

tunnel, url = start_tunnel()
announce(url)
print("Live engine log. Keep this cell running while generating images.\n" + "-" * 60)

# 4) Stream engine log
pos = 0
try:
    while True:
        with open(COMFY_LOG, "rb") as f:
            f.seek(pos); chunk = f.read(); pos = f.tell()
        if chunk:
            sys.stdout.write(chunk.decode(errors="replace")); sys.stdout.flush()
        if backend.poll() is not None:
            print("\n❌ Generation engine stopped, re-run this cell."); break
        if flask_proc.poll() is not None:
            print("\n❌ Web UI stopped, re-run this cell."); break
        if tunnel.poll() is not None:
            print("\n⚠️ Tunnel reset. Reconnecting…")
            tunnel, url = start_tunnel(); announce(url)
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopped streaming logs. The application is still active.")

### 💡 Prompting & Generation Tips

- **Rendering Text in Images:** Wrap the exact text inside double quotes (e.g. `a storefront sign that says "MOSS COFFEE"`). The model is trained to generate English and Chinese characters accurately when placed in quotes.
- **Checkpoint Selection:**
  - **MOSS-Image SFT:** Best for photorealism, detailed natural textures, and strict adherence to long prompts.
  - **Kromoss:** Best for artistic illustrations, high-contrast cinematic scenes, and stylized character portraits.
  - **MOSS-Image Base:** The non-aligned base weights; ideal for broad stylistic experiments.
- **Using the 8-Step Turbo LoRA:**
  - Select `8-steps-dmd-lora.safetensors` in **LoRA 1** with strength `1.0`.
  - Set **Steps** to `8`, **CFG** to `1.0`, and turn **Skip-Layer Guidance (SLG) off** for fast outputs.
- **Prompt Upsampler:** Turn this **ON** for short prompts (e.g. `cyberpunk street at night`) so the model enriches them with detailed visual facets. Turn it **OFF** when you have already written an elaborate, self-contained prompt to save generation time.
- **Skip-Layer Guidance (SLG):** Enhances contrast, micro-textures, and fine structural elements during standard 20–25 step sampling.

### 🔗 Supported `EXTRA_LORA_URLS` Formats (v2)
You can paste one or more links separated by commas or spaces:
- **Civitai:** Model page URL (`https://civitai.com/models/12345` or `https://civitai.com/models/12345?modelVersionId=67890`) or direct download link (`https://civitai.com/api/download/models/67890`). Add your `CIVITAI_API_KEY` if the model requires login.
- **Google Drive:** Share link (`https://drive.google.com/file/d/<ID>/view?usp=sharing`), direct link (`https://drive.google.com/uc?id=<ID>`), or shared folder link (`https://drive.google.com/drive/folders/<ID>`).
- **Hugging Face:** Direct `/resolve/` or `/blob/` link (`https://huggingface.co/user/repo/resolve/main/lora.safetensors`).
- **Custom Filename (Optional):** Prefix any URL with `my_name.safetensors|https://...` to force a specific filename.

### 📦 `EXTRA_MODEL_URLS` (v3)
Accepts the exact same link formats as `EXTRA_LORA_URLS` (Civitai, Google Drive, Hugging Face, direct, `name|url`). Files are saved to `ComfyUI/models/diffusion_models` and show up in the **Checkpoint** dropdown. Only MOSS-Image–architecture diffusion models (UNet-only weights) will work.

- **Embedded PNG Metadata:** All images saved in `output` or downloaded via the UI now embed the full prompt (including the upsampled caption), negative prompt, seed, sampler, CFG, LoRAs, and ComfyUI workflow. You can drag and drop any generated PNG into Civitai, Automatic1111 PNG Info, or ComfyUI to read all generation settings.

- **8-Step Turbo LoRA (v4):** `DOWNLOAD_TURBO_LORA` is independent of `DOWNLOAD_KROMOSS`. You can leave `DOWNLOAD_KROMOSS = False` to save ~5.16 GB of disk space and download time while still using the Turbo LoRA with MOSS-Image SFT or Base.